<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/singlecell_Gene_Expression_Disease_Analysis_Untitled_(Under_The_Guidence_IITB_IC_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing necessary libraries and suppressing warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.switch_backend('Agg')

%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

print('Libraries imported successfully.')

In [ ]:
# Load the dataset from CSV
file_path = '/content/Gene_Expression_Analysis_and_Disease_Relationship_Synthetic.csv'

try:
    df = pd.read_csv(file_path, encoding='ascii')
    print('Data loaded successfully.')
except Exception as e:
    print('Error loading the data:', e)
    # Note: This error handling is critical for users who might use different file paths or encodings.

In [ ]:
print('First five rows of the dataset:')
print(df.head())

print('\nDataset Information:')
print(df.info())

print('\nStatistical Summary:')
print(df.describe())

# Check for missing values
print('\nMissing Values in Each Column:')
print(df.isnull().sum())

In [ ]:
# Check for missing values and remove or impute if necessary
if df.isnull().sum().sum() > 0:
    print('Missing values detected. Proceeding with imputation or removal.')

    df = df.dropna()
else:
    print('No missing values detected.')

df['Cell_Type'] = df['Cell_Type'].astype('category')
df['Disease_Status'] = df['Disease_Status'].astype('category')

# For modeling, we need to encode the target variable, Disease_Status.
le = LabelEncoder()
df['Disease_Status_Encoded'] = le.fit_transform(df['Disease_Status'])

print('Data preprocessing complete.')

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Cell_Type', data=df, palette='viridis')
plt.title('Distribution of Cell Types')
plt.xlabel('Cell Type')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.countplot(x='Disease_Status', data=df, palette='magma')
plt.title('Distribution of Disease Status')
plt.xlabel('Disease Status')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Histograms for numeric columns
numeric_cols = ['Gene_E_Housekeeping', 'Gene_A_Oncogene', 'Gene_B_Immune', 'Gene_C_Stromal', 'Gene_D_Therapy', 'Pathway_Score_Inflam', 'UMAP_1']
df[numeric_cols].hist(figsize=(12, 10), bins=20, edgecolor='black')
plt.tight_layout()
plt.show()

# Pair Plot to see relationships among numeric variables
sns.pairplot(df[numeric_cols])
plt.show()

numeric_df = df[numeric_cols]
if len(numeric_df.columns) >= 4:
    plt.figure(figsize=(10, 8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap of Numeric Features')
    plt.show()

# Boxen Plot for gene expression values for a more detailed distribution view
plt.figure(figsize=(12, 6))
sns.boxenplot(data=df[numeric_cols[:-1]], palette='Set2')
plt.title('Boxen Plot of Gene Expression Values')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
predictor_cols = ['Gene_E_Housekeeping', 'Gene_A_Oncogene', 'Gene_B_Immune', 'Gene_C_Stromal', 'Gene_D_Therapy', 'Pathway_Score_Inflam']
X = df[predictor_cols]
y = df['Disease_Status_Encoded']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print('Prediction Accuracy:', accuracy)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Detailed classification report
print('\nClassification Report:')
print(classification_report(y_test, y_pred))